# Singularity RWKV - Colab Egitim Not Defteri

Hedef: ~450M parametrelik RWKV kod modeli (T4, bf16 + 8-bit AdamW + checkpoint).
Diller: Python, C++, JavaScript, TypeScript, Java, C#, Go (7 dil).
Filtre: 32-4000 bayt, dedup, dil, C++ -> g++ derleme kontrolu, diger -> sentaks dogrulama.
Cikti: weights.bin -> yerelde `./singularity --chat weights.bin "..."` ile calistirilir.

## 0. Runtime
Runtime -> Change runtime type -> **GPU** (T4) sec. Ucretsiz katmanda bazen P100 gelir; bf16 yine calisir.

In [ ]:
!pip install bitsandbytes datasets -q

## 1. Kodlari getir
En temiz yontem: `train/` klasorunu bir GitHub repoya yukle, buradan clone et.
Alternatif: sol dosya panelinden yukle, ya da Google Drive'a birakip asagidaki gibi kopyala.

In [ ]:
# GitHub'dan (en temiz):
!git clone https://github.com/<SEN>/singularity-train.git
%cd singularity-train/train

# VEYA Drive'dan:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/singularity-train/train /content/train
# %cd /content/train

## 2. Veri hazirla (kuratoer botu)
Yerel kod klasorunuzu yuklediyseniz topla, VEYA HuggingFace'den (code_search_net) al.

In [ ]:
# Yerel kod -> kuratolu korpus (API gerektirmez):
# !python curate.py --roots /content/code \
#     --langs python cpp javascript typescript java csharp go \
#     --out curated

# HuggingFace'den codu cek (token GEREKMEZ, acik datasetler):
# Kaynaklar: m-a-p/CodeFeedback-Filtered-Instruction (lang alani, 7 dil hepsi)
#          + codeparrot/codeparrot-clean (Python icin ek)
# max_per_lang: her dilden kac ornek. 2000-5000 iwiyetli bir baslangic.
!python curate.py --hf \
    --langs python cpp javascript typescript java csharp go \
    --max_per_lang 2000 \
    --out curated

# UYARI: javascript/csharp dataset'in basinda az oldugu icin tarama
# birkac dakika surebilir. Sabirla bekle (Colab'da sorun degil).

In [ ]:
# (Opsiyonel) LLM ile sentetik ornek uret - anahtari gir, provider sec.
import os, getpass
provider = "openrouter"   # veya "groq"
key = getpass.getpass(f"{provider} API anahtari: ")
os.environ['OPENROUTER_API_KEY' if provider == "openrouter" else 'GROQ_API_KEY'] = key
# NOT: provider ve model curate.py icinde degisir --api ve --model ile verilir.
# Asagidaki komutu kendi provider/model tercihine gore duzenleyip yorumdan cikar.
# !python curate.py --generate 200 --api openrouter --model tencent/hy3:free --out curated
# !python curate.py --generate 200 --api groq --model llama-3.3-70b-versatile --out curated

## 3. Egit (450M, T4 icin guvenli parametreler)

In [ ]:
!python train.py \
    --roots_code curated/python curated/cpp curated/javascript curated/typescript curated/java curated/csharp curated/go \
    --roots_text curated/txt \
    --C 1536 --n_blocks 24 --seq_len 768 --batch 4 \
    --code_ratio 0.8 --use_8bit --checkpoint \
    --out /content/drive/MyDrive/singularity/weights.bin

In [ ]:
# Drive yoksa yerel yazip indir:
# !python train.py --roots_code curated/python ... --out weights.bin
# from google.colab import files
# files.download('weights.bin')

## 4. Sonraki adimlar
- Session koparsa Drive'daki son weights.bin'i indir, yeni session'da ayni komutla devam et.
- Yetmezse C=1024 n_blocks=24 (~200M) dene (daha guvenli).
- Kalite icin curate.py ile daha cok/kaliteli veri topla veya LLM generate ile zenginlestir.